# Extract SYMP Ontology Terms

Reads the raw Symptom Ontology JSON dump (`../data/symp-ontology.json`)
and produces a flat JSON-Lines file of `(id, label)` pairs that the
downstream RAG notebook consumes as `../data/onto-symp-labels.jsonl`.


## 1. Helper: parse a SYMP CURIE from an OBO URL

In [ ]:
def extract_symp_id(url):
    """Convert an OBO PURL into the compact SYMP CURIE.

    Example:
        'http://purl.obolibrary.org/obo/SYMP_0000000' -> 'SYMP:0000000'
    """
    tail = url.rsplit("/", 1)[-1]      # e.g. 'SYMP_0000000'
    prefix, num = tail.split("_", 1)   # ('SYMP', '0000000')
    return f"{prefix}:{num}"


## 2. Load the ontology JSON and pull out class-level terms

We keep only `CLASS` nodes that have both an `id` and a human-readable
`lbl`, since later stages match on the natural-language label.


In [ ]:
import json

with open("../data/symp-ontology.json") as f:
    data = json.load(f)

# OBO JSON puts everything under graphs[0].nodes
nodes = data["graphs"][0]["nodes"]

terms = [
    (extract_symp_id(n["id"]), n["lbl"])
    for n in nodes
    if n.get("type") == "CLASS" and "lbl" in n and "id" in n
]


## 3. Sanity check — peek at the first few terms

In [3]:
terms[:5]


[('SYMP:0000000', 'cellulitis'),
 ('SYMP:0000001', 'abdominal cramp'),
 ('SYMP:0000002', 'abdominal distention'),
 ('SYMP:0000003', 'obsolete acute enteritis in newborns'),
 ('SYMP:0000004', 'obsolete arrested moulting')]

## 4. Write the flat JSONL the rest of the pipeline consumes

Each line is `{"id": "SYMP:xxxxxxx", "label": "..."}`. This is the
exact format `load_onto_symp_labels` / `load_onto_symp_terms` expect
in `rag-sym-semantic-match-vllm.ipynb`.


In [ ]:
# NOTE: filename matches what the RAG notebook reads in.
with open("../data/onto-symp-labels.jsonl", "w") as f:
    for term_id, label in terms:
        f.write(json.dumps({"id": term_id, "label": label}) + "\n")
